# Mooring Field Detection — Kaggle GPU Coastal Scan

Run YOLO boat detection on **already-fetched** satellite tiles using a free **GPU T4**.
Code loads from **GitHub**; tiles + weights come from a Kaggle Dataset you upload.

## Before you open this notebook

**On your PC (repo root):**

```bash
# 1. Candidates (Cape Cod example — full peninsula bbox)
python -m mooring_fields.cli generate-candidates \
  --bbox -70.75,41.50,-69.90,42.10 --types MO,M \
  --max-sites 160 --out data/candidates_CapeCod.kml

# 2. Fetch tiles only (Google Static Maps — stays in free-tier if sites*5 <= 800)
python -m mooring_fields.cli fetch-scan \
  --kml data/candidates_CapeCod.kml --max-requests 800

# 3. Zip only this KML's tiles + model weights
python -m mooring_fields.cli package-kaggle-scan \
  --kml data/candidates_CapeCod.kml --out kaggle_scan_payload.zip

# 4. Commit + push this repo to GitHub (so Cell 1 can clone the latest code)
git add -A && git commit -m "..." && git push
```

**On Kaggle:**

1. Create Dataset from `kaggle_scan_payload.zip` (name e.g. `mooring-scan-capecod`)
2. New notebook → **GPU T4**, Internet **On**
3. **Add data** → attach that dataset
4. File → Import this notebook (or paste cells) → run in order
5. Save Version → download `scan_out/mooring_fields.db`

**Back on PC:**

```bash
python -m mooring_fields.cli import-scan --from-db path/to/mooring_fields.db
python -m mooring_fields.cli enrich-all --only-new
# hard-refresh http://127.0.0.1:5173
```

In [ ]:
# Cell 1 — Clone repo from GitHub + install
import subprocess, sys, shutil, os
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
# Must match the repo you pushed (branch main)
URL = "https://github.com/IshanKasam/MooringFieldDetection.git"

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "python-dotenv"],
    check=True,
)

import torch
from mooring_fields.runtime import cuda_available

print("repo:", REPO)
print("cuda:", cuda_available(), torch.cuda.get_device_name(0) if cuda_available() else None)
print(
    "/kaggle/input:",
    sorted(p.name for p in Path("/kaggle/input").iterdir())
    if Path("/kaggle/input").exists()
    else None,
)
assert cuda_available(), "Settings → Accelerator → GPU T4 (or P100)"

In [ ]:
# Cell 2 — Find uploaded payload + copy into writable work dir
import json, os, sys
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

from mooring_fields.kaggle_scan import materialize_kaggle_scan_input

INPUT = Path("/kaggle/input")
payload = None
for p in sorted(INPUT.iterdir()) if INPUT.exists() else []:
    if (p / "candidates.kml").is_file() or list(p.rglob("candidates.kml")):
        payload = p
        break

assert payload is not None, (
    "Attach your mooring-scan-* dataset (must contain candidates.kml at top level or nested)."
)

layout = materialize_kaggle_scan_input(payload)
print(json.dumps(layout, indent=2))
assert layout["png_count"] > 0, layout
assert layout["weights"], (
    "Payload missing weights/best.pt — re-run package-kaggle-scan without --no-weights"
)

In [ ]:
# Cell 3 — GPU detection (NO Google fetch; tiles already in the payload)
import json, os, sys
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

from mooring_fields.cli import scan_cmd
from mooring_fields.kaggle_scan import materialize_kaggle_scan_input

INPUT = Path("/kaggle/input")
payload = next(
    p
    for p in INPUT.iterdir()
    if (p / "candidates.kml").is_file() or list(p.rglob("candidates.kml"))
)
layout = materialize_kaggle_scan_input(payload)

out_dir = Path("/kaggle/working/scan_out")
out_dir.mkdir(parents=True, exist_ok=True)

scan_cmd([
    "--kml", layout["kml"],
    "--skip-fetch",
    "--imagery-dir", layout["imagery_dir"],
    "--weights", layout["weights"],
    "--db", str(out_dir / "mooring_fields.db"),
    "--output-dir", str(out_dir),
])

db = out_dir / "mooring_fields.db"
assert db.is_file(), "scan did not write mooring_fields.db"
print("DOWNLOAD THIS FILE →", db)
print("Optional KML →", out_dir / "discovered_fields.kml")

## After Save Version

1. Open the version **Output** tab → download `scan_out/mooring_fields.db`
2. Locally:
   ```bash
   python -m mooring_fields.cli import-scan --from-db ~/Downloads/mooring_fields.db
   python -m mooring_fields.cli enrich-all --only-new
   ```
3. Hard-refresh the web app (`Ctrl+Shift+R`) — new Cape Cod dots should appear.

No Google Maps calls happen on Kaggle in this notebook (fetch was local).